## Adaptive pretraining

### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

### Key Imports

In [ ]:
import pandas as pd
import torch

from config import APT, APT_EPOCHS, IDIOMS, RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.apt_pools import build_pools, build_tapt_pool
from data.loader_twd_labelled import load_splits
from models.apt import adapt
from models.frozen_probe import probe
from models.plm_finetune import finetune
from sklearn.metrics import f1_score

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

### Arms

In [ ]:
# arm -> (pool builder, epochs key, starting checkpoint, seed-dependent)
VANILLA = SHAH_PLM[ENC]["model_name"]

ARMS = {
    "dapt-fomc": (lambda seed: build_pools(verbose=False)[0], "dapt", VANILLA, False),
    "dapt-global": (lambda seed: build_pools(verbose=False)[1], "dapt", VANILLA, False),
    "tapt": (lambda seed: load_splits("benchmark", seed=seed)[0], "tapt", VANILLA, True),
    "curated-tapt": (build_tapt_pool, "curated-tapt", VANILLA, True),
    "dapt-global+curated-tapt": (
        build_tapt_pool,
        "curated-tapt",
        str(RESULTS_DIR / "models" / "dapt-global"),
        True,
    ),
}

### Continued pretraining

In [ ]:
for arm, (pool_fn, epochs_key, start, per_seed) in ARMS.items():
    for seed in SEEDS if per_seed else [None]:
        name = f"{arm}-s{seed}" if per_seed else arm
        save_dir = str(RESULTS_DIR / "models" / name)
        if os.path.isdir(save_dir):
            print(f"{name}: already adapted, skipping")
            continue
        sentences = pool_fn(seed)["sentence"].to_list()
        print(f"{name}: {len(sentences):,} sentences", flush=True)
        adapt(
            sentences,
            model_name=start,
            epochs=APT_EPOCHS[epochs_key],
            save_dir=save_dir,
            device=DEVICE,
            verbose=True,
            **APT,
        )

### Masked-idiom probe

In [ ]:
from transformers import pipeline


def probe_idioms(model_path):
    mlm = pipeline("fill-mask", model=model_path, device=0 if DEVICE == "cuda" else -1)
    mask = mlm.tokenizer.mask_token
    rows = []
    for phrase, gold in IDIOMS:
        top5 = [
            r["token_str"].strip().lower()
            for r in mlm(phrase.replace("[MASK]", mask), top_k=5)
        ]
        rows.append(
            dict(phrase=phrase, gold=gold, hit=gold.lower() in top5, top5="|".join(top5))
        )
    del mlm
    torch.cuda.empty_cache()
    return rows


models = {ENC: VANILLA}
for arm, (_, _, _, per_seed) in ARMS.items():
    name = f"{arm}-s{SEEDS[0]}" if per_seed else arm
    models[arm] = str(RESULTS_DIR / "models" / name)

records = []
for label, path in models.items():
    rows = probe_idioms(path)
    records += [dict(model=label, **r) for r in rows]
    print(f"{label}: {sum(r['hit'] for r in rows)}/{len(rows)}", flush=True)

idf = pd.DataFrame(records)
idf.to_csv(RESULTS_DIR / "idioms.csv", index=False)
print("saved ->", RESULTS_DIR / "idioms.csv")

### Fine-tune

In [ ]:
cfg = SHAH_PLM[ENC]

for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=str(RESULTS_DIR / "models" / name),
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{model_key} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()

### Frozen probe

In [ ]:
for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"frozen-{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        pred = probe(
            train,
            test,
            model_name=str(RESULTS_DIR / "models" / name),
            device=DEVICE,
            seed=seed,
        )
        true = test["label"].to_list()
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs="",
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{model_key} seed {seed}: macro={f1_score(true, pred, average='macro'):.4f}")
